In [ ]:
# ==========================================
# Cell 1: 基础配置 (V3.0.0 - Global + Kalman + CNN-BiLSTM)
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data' 

OUT_DIR = ROOT / 'output_300'
CM_DIR = OUT_DIR / 'confusion_matrices'
MODEL_DIR = OUT_DIR / 'models'

for d in [CM_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Base Dir: {OUT_DIR}')

In [ ]:
# ==========================================
# Cell 2: 数据读取与 Kalman Filter 核心逻辑
# ==========================================
stations_list_path = DATA_DIR / 'stations_list'
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
TOTAL_STATIONS = len(station_ids)

def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = load_labels(DATA_DIR / 'PEMS_testlabels')

# --- 核心：一维卡尔曼滤波函数 (1D Kalman Filter) ---
def apply_kalman_filter(curve, process_variance=1e-4, measurement_variance=1e-2):
    """使用卡尔曼滤波对交通流一维时序数据进行软去噪"""
    n = len(curve)
    xhat = np.zeros(n)      # 滤波后的最优估计
    P = np.zeros(n)         # 估计误差协方差
    xhat[0] = curve[0]
    P[0] = 1.0
    
    for k in range(1, n):
        # 预测步骤
        xhat_minus = xhat[k-1]
        P_minus = P[k-1] + process_variance
        # 更新步骤
        K = P_minus / (P_minus + measurement_variance) # 卡尔曼增益
        xhat[k] = xhat_minus + K * (curve[k] - xhat_minus)
        P[k] = (1 - K) * P_minus
        
    return xhat

In [ ]:
# ==========================================
# Cell 3: 全局数据加载器 (不再分组，全量数据)
# ==========================================
def _process_curve_with_kalman(curve: np.ndarray):
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    
    # 填充缺失值
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any(): 
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)
    
    # --- 引入卡尔曼滤波去噪 ---
    curve = apply_kalman_filter(curve)

    # 归一化与特征提取
    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    mean_val, std_val = np.mean(norm_curve), np.std(norm_curve)
    log_max_val = np.log1p(max_val) / 10.0 
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    grad_curve = np.gradient(norm_curve)
    return np.stack([norm_curve, grad_curve], axis=0).astype(np.float32), stats

class PEMS_GlobalDataset(Dataset):
    def __init__(self, split_path, labels, augment=False, split_name=""):
        self.samples, self.augment, self.split_name = [], augment, split_name

        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            
            # 【安全校验】：跳过缺失站点的残缺天数
            if mat is None or mat.shape[0] != TOTAL_STATIONS or mat.shape[1] < 144: 
                continue
                
            y = int(labels[day_i]) - 1
            if y < 0 or y > 6: continue

            for local_sidx in range(TOTAL_STATIONS):
                two_channel_seq, stats = _process_curve_with_kalman(mat[local_sidx, :144])
                if not np.isfinite(two_channel_seq).all(): continue
                meta = {
                    "station_id": int(station_ids[local_sidx]),
                    "station_pos": int(local_sidx),
                    "day_i": int(day_i), "split": self.split_name,
                }
                self.samples.append((two_channel_seq, stats, y, meta))
                
        self.n = len(self.samples)
        print(f"[{split_name}] 加载完成，共有 {self.n} 条独立时序样本。")

    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, stats, y, meta = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)
        if self.augment:
            seq_tensor = seq_tensor * random.uniform(0.95, 1.05)
            shifts = random.randint(-2, 2)
            if shifts != 0: seq_tensor = torch.roll(seq_tensor, shifts=shifts, dims=1)
            seq_tensor += torch.randn_like(seq_tensor) * 0.005
        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long), meta

In [ ]:
# ==========================================
# Cell 4: 1D-CNN + BiLSTM 模型架构
# ==========================================
class CNN_BiLSTM_Model(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super().__init__()
        # 1. CNN 提取局部特征并降维 (144 -> 36)
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout1d(0.1),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout1d(0.1)
        )
        
        # 2. BiLSTM 提取时序记忆规律
        self.lstm = nn.LSTM(
            input_size=64, hidden_size=64, 
            num_layers=2, batch_first=True, bidirectional=True, dropout=0.2
        )
        
        # 3. 注意力/池化与分类
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3), 
            nn.Linear(128 + 3, 64), # 128 (BiLSTM_hidden*2) + 3 (stats)
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        # x shape: (Batch, 2, 144)
        cnn_out = self.cnn(x) # shape: (Batch, 64, 36)
        
        # 调整维度以适配 LSTM: (Batch, Seq_Len, Features) -> (Batch, 36, 64)
        lstm_in = cnn_out.permute(0, 2, 1) 
        
        lstm_out, _ = self.lstm(lstm_in) # shape: (Batch, 36, 128)
        
        # 调整回 (Batch, Features, Seq_Len) 进行全局池化
        lstm_out_pool = lstm_out.permute(0, 2, 1) # shape: (Batch, 128, 36)
        features = self.gap(lstm_out_pool).view(lstm_out_pool.size(0), -1) # (Batch, 128)
        
        # 拼接统计特征
        combined = torch.cat([features, stats], dim=1) 
        return self.fc(combined)

In [ ]:
# ==========================================
# Cell 5: 构建全局 DataLoader 并执行训练
# ==========================================
train_path = DATA_DIR / 'PEMS_train'
test_path  = DATA_DIR / 'PEMS_test'

print("\n>>> 开始加载全量数据集 (约需要1-2分钟，应用卡尔曼滤波中)...")
ds_train_source = PEMS_GlobalDataset(train_path, labels_train, augment=True, split_name="train_full")
ds_val_source   = PEMS_GlobalDataset(train_path, labels_train, augment=False, split_name="val_clean")
ds_test         = PEMS_GlobalDataset(test_path, labels_test, augment=False, split_name="test")

# 全局 80/20 划分
total_len = len(ds_train_source)
indices = torch.randperm(total_len, generator=torch.Generator().manual_seed(42)).tolist()
train_subset = Subset(ds_train_source, indices[:int(total_len * 0.8)])
val_subset   = Subset(ds_val_source, indices[int(total_len * 0.8):])

loaders = {
    'train': DataLoader(train_subset, batch_size=256, shuffle=True, num_workers=0), # 全局数据可适当调大 batch_size
    'val':   DataLoader(val_subset, batch_size=512, shuffle=False, num_workers=0),
    'test':  DataLoader(ds_test, batch_size=512, shuffle=False, num_workers=0),
}

def train_global_model(loaders, epochs=60):
    model = CNN_BiLSTM_Model().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    weights = torch.tensor([1.0, 2.0, 4.0, 2.0, 1.5, 1.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    best_acc, best_state, counter, patience = 0.0, None, 0, 15
    
    print("\n>>> 开始训练大一统全局模型 (CNN + BiLSTM)...")
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y, _ in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, stats)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step() 
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, stats, y, _ in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                preds = torch.argmax(model(x, stats), dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        if val_acc > best_acc:
            best_acc, best_state, counter = val_acc, model.state_dict(), 0
        else:
            counter += 1
            
        if epoch % 5 == 0:
            print(f"[Global Model] Ep {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%} | LR: {scheduler.get_last_lr()[0]:.6f}")
        
        if counter >= patience:
            print(f"[Global Model] 早停触发于 epoch {epoch}")
            break
            
    return best_state, best_acc

global_model_state, final_acc = train_global_model(loaders, epochs=60)
print(f"\n全局模型训练完成！最佳验证准确率: {final_acc:.2%}")
if global_model_state:
    torch.save(global_model_state, MODEL_DIR / 'cnn_lstm_v300_global.pth')

In [ ]:
# ==========================================
# Cell 6: 全局可视化与结果导出
# ==========================================
def evaluate_global_model(model_state, loader, mode='test'):
    model = CNN_BiLSTM_Model().to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    y_true, y_pred, metas = [], [], []
    with torch.no_grad():
        for x, stats, y, meta in loader:
            x, stats = x.to(DEVICE), stats.to(DEVICE)
            logits = model(x, stats)
            preds = torch.argmax(logits, dim=1)
            
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            if isinstance(meta, dict):
                bs = len(y)
                for i in range(bs):
                    metas.append({k: meta[k][i] for k in meta})

    # 绘制全局混淆矩阵
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
    
    plt.figure(figsize=(10, 8))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f"Global CNN-BiLSTM-Kalman ({mode.capitalize()} Set)", fontsize=16, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    
    save_path = CM_DIR / f'global_{mode}_matrix_v300.png'
    plt.savefig(save_path, dpi=300)
    print(f"\n{mode.capitalize()} 集混淆矩阵已保存: {save_path}")
    plt.close()
    
    # 导出 CSV
    rows = []
    for i in range(len(y_true)):
        m = metas[i]
        yt, yp = y_true[i], y_pred[i]
        rows.append({
            "station_pos": m.get("station_pos", -1),
            "day_i": m.get("day_i", -1), 
            "y_true": yt, "y_pred": yp,
            "true_label": labels[yt], "pred_label": labels[yp],
            "correct": (yt == yp),
        })

    df = pd.DataFrame(rows).sort_values(["correct","station_pos","day_i"]).reset_index(drop=True)
    out_all = CM_DIR / f"predictions_{mode}_global_v300.csv"
    df.to_csv(out_all, index=False, encoding="utf-8-sig")
    print(f"{mode.capitalize()} 集准确率: {df['correct'].mean():.2%}")

evaluate_global_model(global_model_state, loaders['val'], 'val')
evaluate_global_model(global_model_state, loaders['test'], 'test')